# conv-stride-downsample — worked example 3: Two stacked strided convs multiply the downsample factor

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-stride-downsample`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Stacking convolutions multiplies their stride factors: a stride-2 layer followed by a stride-2 layer downsamples by roughly 4 overall. Each layer applies `out = (in - K) // S + 1` to whatever its predecessor produced, so you chain the formula layer by layer to get the final spatial size.

## Worked solution

**Goal.** Given an input length and two conv layers each described by `(K, S)`, predict the final length and verify against a real two-layer `Sequential`.

**Step 1 — apply the formula to layer 1.** With `L0=64`, layer 1 `(K=3, S=2)`: `L1 = (64 - 3) // 2 + 1 = 61 // 2 + 1 = 30 + 1 = 31`.

**Step 2 — feed L1 into layer 2.** Layer 2 `(K=3, S=2)` sees an input of length `31`: `L2 = (31 - 3) // 2 + 1 = 28 // 2 + 1 = 14 + 1 = 15`. So 64 collapses to 15 — close to but not exactly 64/4=16, because each floor division shaves the ragged edge.

**Step 3 — why it is not exactly 4x.** Each layer independently drops the partial window at its trailing edge before the next layer sees it. These roundings compound, so the effective downsample is approximately the product of strides (4x) but biased slightly smaller. Predicting it requires actually chaining the formula, not just dividing by 4.

**Step 4 — verify.** We build `Sequential(Conv1d, Conv1d)` with the same `(K, S)` per layer and assert the final length matches our chained prediction.

In [ ]:
def chained_conv1d_outlen(l_in, layers):
    l = l_in
    for k, s in layers:
        l = (l - k) // s + 1
    return l

t.manual_seed(0)
C = 4
l_in = 64
layers = [(3, 2), (3, 2)]
x = t.randn(1, C, l_in)
net = t.nn.Sequential(
    t.nn.Conv1d(C, C, kernel_size=layers[0][0], stride=layers[0][1]),
    t.nn.Conv1d(C, C, kernel_size=layers[1][0], stride=layers[1][1]),
)
y = net(x)
pred = chained_conv1d_outlen(l_in, layers)
print('predicted final len:', pred)
print('actual    final len:', y.shape[-1])
print('naive /4 would say:', l_in // 4)